# Lesson 2: solving Markov chains

This lesson can be downloaded as a notebook, a notebook for colab and a python file [here](https://marmote.gitlabpages.inria.fr/marmote/python_downloads.html)

This C++ notebook mirrors the Python lesson and focuses on the main solution objects returned by Marmote.

**Import the modules**

In [1]:
#ifdef _WIN32
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteCore")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteMarkovChain")
#pragma cling add_library_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/bin")
#pragma cling load("marmoteCore.dll")
#pragma cling load("marmoteMarkovChain.dll")
#else
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMarkovChain")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMarkovChain.so")
#endif

In [2]:
// --- Standard C++ utilities used in this notebook ---
#include <iostream>
#include <iomanip>
#include <string>
#include <vector>

// --- Marmote headers used in this lesson ---
#include <marmoteCore/marmoteCore>
#include <marmoteMarkovChain/marmoteMarkovChain>

// --- Convenience declarations for the cells below ---
using namespace std;
using namespace marmote;

In Lesson 1, we saw how to create and inspect Markov chains. We now illustrate the different metrics that can be computed on them in C++.

## First example: solving discrete-time Markov chains

In [3]:
// Recreate the 3-state discrete-time Markov chain of Lesson 1.
double states[3] = {0.0, 1.0, 2.0};
FullMatrix* P = new FullMatrix(3);
P->set_type(DISCRETE);
P->setEntry(0,0,0.25); P->setEntry(0,1,0.50); P->setEntry(0,2,0.25);
P->setEntry(1,0,0.40); P->setEntry(1,1,0.20); P->setEntry(1,2,0.40);
P->setEntry(2,0,0.40); P->setEntry(2,1,0.30); P->setEntry(2,2,0.30);
double initial_prob[3] = {0.2, 0.2, 0.6};
DiscreteDistribution* initial = new DiscreteDistribution(3, states, initial_prob);
MarkovChain* c1 = new MarkovChain(P);
c1->set_init_distribution(initial);
c1->set_model_name("Demo");

### Transient distributions

The method `TransientDistributionDT` is used exactly as in Python.

In [4]:
// Compute transient distributions after 1, 2 and 3 steps.
DiscreteDistribution* pi1 = c1->TransientDistributionDT(1);
DiscreteDistribution* pi2 = c1->TransientDistributionDT(2);
DiscreteDistribution* pi3 = c1->TransientDistributionDT(3);
cout << *pi1 << endl;
cout << *pi2 << endl;
cout << *pi3 << endl;

// Change the initial distribution to a Dirac mass at state 0.
DiracDistribution* pi0_dirac = new DiracDistribution(0.0);
c1->set_init_distribution(pi0_dirac);
cout << *(c1->TransientDistributionDT(1)) << endl;

// Use a uniform discrete distribution instead.
UniformDiscreteDistribution* pi0_uniform = new UniformDiscreteDistribution(0, 2);
c1->set_init_distribution(pi0_uniform);
cout << *(c1->TransientDistributionDT(1)) << endl;

DiscreteDistribution (Object at 0x59dd9cdcfa30)Discrete distribution values { 0  1  2  } probas {     0.37     0.32     0.31 }
DiscreteDistribution (Object at 0x59dd9ce2d7f0)Discrete distribution values { 0  1  2  } probas {   0.3445    0.342   0.3135 }
DiscreteDistribution (Object at 0x59dd9cdd1040)Discrete distribution values { 0  1  2  } probas { 0.348325   0.3347 0.316975 }
DiscreteDistribution (Object at 0x59dd9cd66e80)Discrete distribution values { 0  1  2  } probas {     0.25      0.5     0.25 }
DiscreteDistribution (Object at 0x59dd9ce32d90)Discrete distribution values { 0  1  2  } probas {     0.35 0.333333 0.316667 }


### Stationary distribution

As in Python, we compare the default stationary distribution method, the RLGL variant, and the exact distribution.

In [5]:
// Default stationary distribution.
DiscreteDistribution* pista = c1->StationaryDistribution();
cout << *pista << endl;

// RLGL stationary distribution.
UniformDiscreteDistribution* u0 = new UniformDiscreteDistribution(0, 2);
DiscreteDistribution* pista2 = c1->StationaryDistributionRLGL(100, 1e-10, u0, false);
cout << *pista2 << endl;
cout << "Distance L1 between both approximations = "
     << Distribution::DistanceL1(pista, pista2) << endl;

// Exact stationary distribution.
double prosta_ex[3] = {8.0/23.0, 85.0/253.0, 80.0/253.0};
DiscreteDistribution* pista_ex = new DiscreteDistribution(3, states, prosta_ex);
cout << *pista_ex << endl;
cout << "Distance L1 between default and exact pi = "
     << Distribution::DistanceL1(pista, pista_ex) << endl;

DiscreteDistribution (Object at 0x59dd9ce1e8f0)Discrete distribution values { 0  1  2  } probas { 0.347826 0.335968 0.316206 }
DiscreteDistribution (Object at 0x59dd9ce40650)Discrete distribution values { 0  1  2  } probas { 0.347826 0.335968 0.316206 }
Distance L1 between both approximations = 2.96228e-08
DiscreteDistribution (Object at 0x59dd9ce1e9e0)Discrete distribution values { 0 1 2 } probas { 0.347826 0.335968 0.316206 }
Distance L1 between default and exact pi = 2.96316e-08


### Simulation

Simulation objects in C++ expose the same information as in Python, but vectors are accessed explicitly.

In [6]:
// Simulate a trajectory of 10 steps and keep it in memory.
SimulationResult* simRes = c1->SimulateChainDT(10, false, true, false);
simRes->Diagnose(&cout);
for (auto d : simRes->DT_dates()) {
    cout << d << " ";
}
cout << endl;

// Run a second simulation with occupancy statistics and no stored trajectory.
SimulationResult* simRes2 = c1->SimulateChainDT(10, true, false, true);
DiscreteDistribution* trDis = simRes2->Distribution();
cout << *trDis << endl;

# Simulation result
# Time type: discrete
# Samples collected: 11
# Keeps the trajectory
# DT trajectory size: 11
# CT trajectory size: 0
# Last state: 2
# Last time:  10
0 1 2 3 4 5 6 7 8 9 10 
         0        2 2
         1        1 1
         2        1 1
         3        2 2
         4        0 0
         5        1 1
         6        1 1
         7        0 0
         8        2 2
         9        0 0
        10        1 1
# State   0:   cum time =        4   (40%)
# State   1:   cum time =        4   (40%)
# State   2:   cum time =        3   (30%)
DiscreteDistribution (Object at 0x59dd9ce3fb00)Discrete distribution values { 0  1  2  } probas {      0.4      0.4      0.3 }


## Second example: solving continuous-time Markov chains

In [7]:
// Recreate the continuous-time chain of Lesson 1.
SparseMatrix* Q = new SparseMatrix(6);
Q->set_type(CONTINUOUS);
Q->setEntry(0,1,1.0);
Q->setEntry(0,0,-1.0);
for (stateType i = 1; i < 6; i++) {
    if (i > 0) {
        Q->setEntry(i,0,1.0);
        Q->addToEntry(i,i,-1.0);
    }
    if (i < 5) {
        Q->setEntry(i,i+1,1.0);
        Q->addToEntry(i,i,-1.0);
    }
}
MarkovChain* c2 = new MarkovChain(Q);
c2->set_init_distribution(initial);
c2->set_model_name("Demo_Continuous");

// Stationary distribution and simulation.
DiscreteDistribution* stadis = c2->StationaryDistribution();
cout << *stadis << endl;
SimulationResult* simresCT = c2->SimulateChainCT(10.0, false, true, true, true);
simresCT->Diagnose(&cout);
for (auto t : simresCT->CT_dates()) {
    cout << t << " ";
}
cout << endl;

// Hitting time simulation toward state 5.
bool hitset[6] = {false, false, false, false, false, true};
SimulationResult* hitSim = c2->SimulateHittingTime(static_cast<cardinalType>(0), hitset, 50, 100.0);
for (auto t : hitSim->CT_dates()) {
    cout << t << " ";
}
cout << endl;

// Average hitting times for all starting states.
double* avghit = c2->AverageHittingTimes(hitset);
for (int i = 0; i < 6; i++) {
    cout << avghit[i] << " ";
}
cout << endl;

DiscreteDistribution (Object at 0x59dd9bf46a30)Discrete distribution values { 0  1  2  3  4  5  } probas {      0.5     0.25    0.125   0.0625  0.03125  0.03125 }
[   0]     0.000000        0   0.000000 0
[   1]     0.681791        1   0.681791 1
[   2]     0.807200        2   0.125408 2
[   3]     1.237756        3   0.430556 3
[   4]     1.466771        0   0.229015 0
[   5]     2.528857        1   1.062086 1
[   6]     3.072785        2   0.543928 2
[   7]     5.419894        3   2.347110 3
[   8]     5.500310        0   0.080416 0
[   9]     6.212693        1   0.712383 1
[  10]     8.713008        0   2.500315 0
[  11]     9.012402        1   0.299395 1
[  12]     9.230629        2   0.218227 2
[  13]     9.390851        0   0.160222 0
[  14]    10.000000        1   0.609149 1
# Simulation result
# Time type: continuous
# Samples collected: 15
# Keeps the trajectory
# DT trajectory size: 0
# CT trajectory size: 15
# Last state: 0
# Last time:  10.000000
0.000000 0.681791 0.807200 

In [8]:
// Release the main objects created in this lesson.
delete c2;
delete simRes;
delete simRes2;
delete c1;